In [1]:
import numpy as np
import pandas as pd

In [21]:
def check_empirical_gap(
    attention_csv_path,
    X,
    feature_col="feature",
    score_col="attention_score",
    K_values=(5, 10, 15, 20),
):
    """
    Computes the empirical separation proxy

        G_K = min_{j in T_K} C_j s_j - max_{k not in T_K} C_k s_k,

    where C_j = E[|X_j|] and s_j is the attention rollout score.

    Parameters
    ----------
    attention_csv_path : str
        CSV with d rows, already ranked by attention score.
    X : np.ndarray
        Dataset used to compute attention scores, shape (N, d).
    feature_col : str
        Column name containing feature names.
    score_col : str
        Column name containing attention scores s_j.
    K_values : tuple
        Values of K for which G_K is computed.

    Returns
    -------
    pd.DataFrame
        Table with empirical gap statistics.
    """

    scores_df = pd.read_csv(attention_csv_path)

    if X.shape[1] != len(scores_df):
        raise ValueError(
            f"Dimension mismatch: X has {X.shape[1]} features, "
            f"but CSV has {len(scores_df)} rows."
        )

    # Ensure scores are ranked from largest to smallest
    scores_df = scores_df.sort_values(score_col, ascending=False).reset_index(drop=True)

    # Compute C_j = E |X_j|
    C = np.mean(np.abs(X), axis=0)
    
    scores_df["C_j"] = C
    scores_df["C_j_s_j"] = scores_df["C_j"] * scores_df[score_col]

    results = []

    d = len(scores_df)

    for K in K_values:
        if K <= 0 or K >= d:
            raise ValueError(f"K must satisfy 1 <= K < d. Got K={K}.")

        topK = scores_df.iloc[:K]
        rest = scores_df.iloc[K:]

        min_topK = topK["C_j_s_j"].min()
        max_rest = rest["C_j_s_j"].max()
        G_K = min_topK - max_rest

        weakest_top = topK.loc[topK["C_j_s_j"].idxmin(), feature_col]
        strongest_outside = rest.loc[rest["C_j_s_j"].idxmax(), feature_col]

        results.append({
            "K": K,
            "min_topK_Cs": min_topK,
            "weakest_topK_feature": weakest_top,
            "max_outside_Cs": max_rest,
            "strongest_outside_feature": strongest_outside,
            "G_K": G_K,
            "separated": G_K > 0,
        })

    return pd.DataFrame(results), scores_df

In [3]:
test_data = np.load("data_processed/test_data_scaled.npz")
X_test = test_data["x"]    
scores_path = "Results/attention_scores.csv"

In [22]:
# Example usage
gap_table, augmented_scores = check_empirical_gap(
    attention_csv_path=scores_path,
    X=X_test,
    feature_col="feature",
    score_col="score",
    K_values=(2, 3, 4, 5, 6)
)

print(gap_table)
# gap_table.to_csv("empirical_gap_GK.csv", index=False)
# augmented_scores.to_csv("attention_scores_with_Cj.csv", index=False)

   K  min_topK_Cs  weakest_topK_feature  max_outside_Cs  \
0  2     1.265781  Dimethylsulfone (mM)        1.102979   
1  3     0.954922          Cystine (mM)        1.102979   
2  4     0.954922          Cystine (mM)        1.033103   
3  5     0.954922          Cystine (mM)        0.936423   
4  6     0.469771          Glucose (mM)        0.936423   

               strongest_outside_feature       G_K  separated  
0  Erythrocyte sedimentation rate (mm/h)  0.162802       True  
1  Erythrocyte sedimentation rate (mm/h) -0.148056      False  
2                LDL cholesterol (mg/dl) -0.078181      False  
3               1,5-Anhydrosorbitol (mM)  0.018499       True  
4               1,5-Anhydrosorbitol (mM) -0.466653      False  
